# Lesson 10 - Ejen AI dalam Pengeluaran

Dalam pelajaran ini anda akan belajar tentang **corak pengeluaran** untuk ejen AI menggunakan Microsoft Agent Framework (Python). Kami membincangkan:

- **Kebolehamatan** — menambah penentuan masa dan log kepada interaksi ejen
- **Penilaian** — menggunakan ejen penilai untuk menilai kualiti respons
- **Pengurusan kos** — strategi untuk pengoptimuman token dan pemilihan model

Senario ini adalah **ejen pelancongan** yang membantu pengguna merancang perjalanan, dengan pemantauan dan penilaian yang ditambah di atasnya.


## Persediaan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Pertimbangan Pengeluaran

Memindahkan ejen AI dari prototaip ke pengeluaran memerlukan perhatian teliti kepada tiga tiang:

1. **Kebolehlihatan** — Anda memerlukan ketelusan terhadap apa yang ejen lakukan, berapa lama ia mengambil masa, dan alat mana yang digunakan. Tanpa penjejakan dan pencatatan, penyahpepijatan isu pengeluaran hampir mustahil.

2. **Penilaian** — Pemeriksaan kualiti automatik memastikan respons ejen kekal tepat, lengkap, dan berguna dari masa ke masa. Ejen penilai boleh memberi skor respons berdasarkan kriteria yang ditetapkan.

3. **Pengurusan Kos** — Penggunaan token memberi kesan langsung kepada kos. Strategi seperti pengoptimuman prompt, pemilihan model, dan penampan membantu mengawal perbelanjaan tanpa mengorbankan kualiti.


## Membina Ejen Boleh Pemerhatian

Kami mentakrifkan alat perjalanan dan membalut panggilan ejen dengan penentuan masa supaya kami boleh memantau kelewatan. Dalam produksi, anda akan mengintegrasikan dengan OpenTelemetry atau backend pengesanan yang serupa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Corak Penilaian

Corak pengeluaran biasa adalah menggunakan agen kedua sebagai **penilai**. Penilai ini menilai tindak balas agen utama berdasarkan kriteria yang telah ditetapkan seperti kesempurnaan, ketepatan, dan kebermanfaatan.

Ini membolehkan:
- Pintu kualiti automatik sebelum tindak balas sampai kepada pengguna
- Pengesanan regresi apabila arahan atau model berubah
- Pemantauan berterusan prestasi agen dari masa ke masa


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategi Pengurusan Kos

Mengawal kos adalah kritikal untuk ejen AI pengeluaran. Berikut adalah strategi utama:

| Strategi | Penerangan |
|---|---|
| **Pengoptimuman arahan** | Kekalkan arahan sistem secara ringkas. Keluarkan konteks yang berlebihan untuk mengurangkan token input. |
| **Pemilihan model** | Gunakan model yang lebih kecil dan murah (contohnya GPT-4o-mini) untuk tugasan mudah seperti pengkelasan atau pengekstrakan, dan simpan model yang lebih besar untuk penalaran kompleks. |
| **Caching** | Cache keputusan alat dan pertanyaan kerap untuk mengelakkan panggilan API yang berulang. |
| **Bajet token** | Tetapkan had `max_tokens` untuk mengelakkan jawapan yang panjang secara tidak dijangka. |
| **Pengelompokan** | Kumpulkan beberapa pertanyaan pengguna ke dalam satu panggilan API di mana boleh. |

Dalam amalan, pendekatan berperingkat berfungsi dengan baik: arahkan permintaan mudah kepada model yang cepat dan murah serta tingkatkan hanya pertanyaan kompleks kepada model yang lebih mampu.


## Ringkasan

Dalam pelajaran ini anda belajar cara untuk:

1. **Menambah keberamatan** kepada interaksi ejen dengan penandaan masa dan log, meletakkan asas untuk penjejakan dan pemantauan.
2. **Menilai tindak balas ejen** secara automatik menggunakan ejen penilai yang menilai kesempurnaan, ketepatan, dan kebermanfaatan.
3. **Menguruskan kos** melalui pengoptimuman arahan, pemilihan model, penyimpanan cache, dan bajet token.

Corak pengeluaran ini membantu memastikan ejen AI anda boleh dipercayai, boleh diukur, dan berkesan dari segi kos pada skala besar.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
